In [1]:
import pandas as pd
import numpy as np

from google.colab import files
uploaded = files.upload()

df = pd.read_excel("Online Retail.xlsx")
df = df.drop_duplicates().copy()

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)

print("Dataset shape:", df.shape)

Saving Online Retail.xlsx to Online Retail.xlsx
Dataset shape: (536641, 8)


In [2]:
data = df[
    (df["UnitPrice"] > 0) &
    (df["Quantity"] > 0) &
    (df["InvoiceDate"].notna()) &
    (df["StockCode"].notna())
].copy()

data["Month"] = data["InvoiceDate"].dt.month
data["Hour"] = data["InvoiceDate"].dt.hour
data["DayOfWeek"] = data["InvoiceDate"].dt.dayofweek

print("Rows available for analysis:", len(data))
data.head()

Rows available for analysis: 524878


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Month,Hour,DayOfWeek
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,12,8,2
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,12,8,2
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,12,8,2
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,12,8,2
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,12,8,2


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

features = ["UnitPrice", "Month", "Hour", "DayOfWeek"]
X = data[features]
y = data["Quantity"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("Baseline MAE:", round(mae, 4))
print("Baseline R²:", round(r2, 4))

Baseline MAE: 10.8462
Baseline R²: 0.0042


In [4]:
report = pd.DataFrame({
    "Metric": [
        "Rows used",
        "Number of features",
        "Training rows",
        "Testing rows",
        "Baseline MAE",
        "Baseline R2"
    ],
    "Value": [
        len(data),
        len(features),
        len(X_train),
        len(X_test),
        mae,
        r2
    ]
})

display(report)
report.to_csv("day10_baseline_model_summary.csv", index=False)
print("Day 10 report saved.")

,Metric,Value
0,Rows used,524878.000000
1,Number of features,4.000000
2,Training rows,419902.000000
3,Testing rows,104976.000000
4,Baseline MAE,10.846158
5,Baseline R2,0.004233


Day 10 report saved.


In [6]:
from google.colab import files
files.download("day10_baseline_model_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>